In [1]:
import numpy as np
import pandas as pd

In [6]:
df = pd.read_csv("../../data/processed/team/subway_merged_base.csv")
df

,지하철역,호선명,총승차인원,무임승차인원,총하차인원,무임하차인원,전체승하차,무임승하차,무임승하차비중,ES,EV,TOTAL
0,제기동,1,501317.0,269661.0,511782.0,290055.0,1013099.0,559716.0,55.25,2,3,5
1,종각,1,1112527.0,151760.0,1083408.0,140929.0,2195935.0,292689.0,13.33,2,4,6
2,종로5가,1,712905.0,247967.0,697910.0,240352.0,1410815.0,488319.0,34.61,0,3,3
3,청량리,1,662997.0,280334.0,656727.0,281655.0,1319724.0,561989.0,42.58,6,2,8
4,강남,2,2302759.0,161221.0,2246564.0,140970.0,4549323.0,302191.0,6.64,0,4,4
...,...,...,...,...,...,...,...,...,...,...,...,...
241,청구,"5,6",219899.0,44565.0,211920.0,42532.0,431819.0,87097.0,20.17,10,5,15
242,충무로,"3,4",867520.0,119883.0,893846.0,121889.0,1761366.0,241772.0,13.73,23,4,27
243,충정로,"2,5",417744.0,63616.0,436275.0,64124.0,854019.0,127740.0,14.96,8,4,12
244,태릉입구,"6,7",438855.0,87568.0,443702.0,85169.0,882557.0,172737.0,19.57,24,6,30


### 대상 컬럼 선정 (엘리베이터 EV, 에스컬레이터 ES)

In [27]:
cols=['EV', 'ES']
data = df[cols].replace(0, 1e-9)  # log(0)은 문제가 발생하기 때문에 0이 아닌 아주 작은 숫자로 대체

In [28]:
data

,EV,ES
0,3,2.000000e+00
1,4,2.000000e+00
2,3,1.000000e-09
3,2,6.000000e+00
4,4,1.000000e-09
...,...,...
241,5,1.000000e+01
242,4,2.300000e+01
243,4,8.000000e+00
244,6,2.400000e+01


In [29]:
data['ES'].unique()

array([2.0e+00, 1.0e-09, 6.0e+00, 4.0e+00, 8.0e+00, 9.0e+00, 1.4e+01,
       1.2e+01, 1.1e+01, 5.0e+00, 1.0e+01, 2.0e+01, 3.0e+00, 1.6e+01,
       1.3e+01, 1.8e+01, 7.0e+00, 1.9e+01, 2.3e+01, 2.6e+01, 3.2e+01,
       2.2e+01, 2.7e+01, 3.4e+01, 7.7e+01, 1.7e+01, 2.5e+01, 3.5e+01,
       2.4e+01, 2.1e+01])

In [30]:
data.sum()

EV     922.0
ES    2198.0
dtype: float64

### 표준화 및 비중 계산

In [31]:
p = data / data.sum()  # 각 역이 차지하는 비중 계산, 0~1로 만들었기 때문에 표준화

In [32]:
p

,EV,ES
0,0.003254,9.099181e-04
1,0.004338,9.099181e-04
2,0.003254,4.549591e-13
3,0.002169,2.729754e-03
4,0.004338,4.549591e-13
...,...,...
241,0.005423,4.549591e-03
242,0.004338,1.046406e-02
243,0.004338,3.639672e-03
244,0.006508,1.091902e-02


### 엔트로피 값 계산

In [18]:
m = len(data)  # 지하철 역 개수
k = 1 / np.log(m)  # 정규화 상수
entropy = -k * (p * np.log(p)).sum()  # 엔트로피 구하는 식

### 가중치 산출

In [33]:
diversity = 1 - entropy
weights = diversity / diversity.sum()

print("엔트로피 기반 객관적 가중치 결과")
for col, w in zip(cols, weights):
    print(f"{col}: {w:.4f}")

엔트로피 기반 객관적 가중치 결과
EV: 0.2021
ES: 0.7979


---
### 가중치 산정 방식 : 엔트로피 기반 역가중치

> 데이터의 객관성을 확보하기 위해 엔트로피 가중치법을 선행 검토한 후, 이를 보완한 역가중치 모델을 최종 채택했다.

1. 엔트로피 가중치 산정 결과
    * EV (엘리베이터) : 0.2021
    * ES (에스컬레이터) : 0.7979

> 엔트로피 기법은 데이터의 변동성이 클수록 높은 가중치를 부여한다. 서울시 지하철 내 에스컬레이터는 설치 대수의 편차가 매우 크기 때문에(최소 0대 ~ 최대 77대) 수학적으로 높은 변별력을 갖는 것으로 분석되었다.

<br>

2. 수리적 가중치의 한계와 가치 충돌
    * 복지 가치 무시 : 거동이 불편한 교통약자에게 엘리베이터는 이동을 위한 유일한 수단인 반면, 에스컬레이터는 보조적 수단이다.
    * 안정적 공급의 역설 : 엘리베이터는 정책적으로 전 역에 고르게 설치되어 있어 데이터의 변동성이 낮다. 이 안정적 공급이 오히려 낮은 가중치로 평가되는 모순이 발생한다.
      
<br>

3. 최종 가중치 결정 : 역가중치 적용

> 엔트로피 법칙 가중치(1:4)대로가 아닌, 데이터의 필수성을 기준으로 재산정하였습니다.

| 구분 | 엘레베이터(EV) | 에스컬레이터(ES) |
| :--- | :--- | :--- |
| 엔트로피 가중치 | 0.2021 (변동성 낮음) | 0.7979 (변동성 높음) |
| 적용 가중치 | 4 (필수) | 1 (보조) |

* 근거

> 데이터의 변동성이 적은 시설(EV)일수록 모든 역에 필수적으로 존재해야 하는 기초 복지 자원임을 의미한다. 따라서 수리적 엔트로피 결과와 반비례하는 역가중치 원리를 적용하여, 교통약자가 체감하는 이동 편의성에 기반한 4:1 가중치를 최종 확정한다. 